In [ ]:
#Q.8
# 요건: web_logs를 session_id 단위로 집계해 view/cart/purchase에 도달한 세션 수를 산출함
# view -> cart, cart-> purchase 단계별 전환율과 정규 경로 전체 전환율을 계산
# 가장 큰 이탈단계와 개선 실험 아이디어, 퍼널 도달표 (단계별 세션수, 전환율)

#포인트 1: 로그는 사이트 한정기록이므로 orders를 web채널로 한정해야됨 
#포인트 2: 고객 X 날짜 병합해서 로그에만 있는 구매 vs 주문에만 있는 구매를 대조


In [ ]:
import pandas as pd

# web_logs는 100만행이므로 usecols/dtypes 지정
path = "../data/web_logs.csv"
usecols = ["session_id", "event_type"]
dtype = {"session_id": "string", "event_type": "category"}

logs = pd.read_csv(path, usecols=usecols, dtype=dtype)
print(logs.shape)
logs.head()


path = "../data/web_logs.csv"
usecols = ["session_id", "event_type"]  

#dtype은 어떤 자료형으로 저장할지
dtype = {
    "session_id" : "string",
    "event_type" : "category"
}



(1000000, 2)


,event_type,session_id
0,view,sess_001e5511
1,purchase,sess_b60fb67c
2,view,sess_d09e432b
3,view,sess_5cc9f9b6
4,view,sess_029b23c8


In [11]:
logs['event_type']

0             view
1         purchase
2             view
3             view
4             view
            ...   
999995      search
999996        view
999997        view
999998        view
999999        view
Name: event_type, Length: 1000000, dtype: category
Categories (4, str): ['cart', 'purchase', 'search', 'view']

In [18]:
path = "../data/web_logs.csv"

df = pd.read_csv(path)
columns = pd.read_csv(path, nrows=0).columns
print(columns.tolist())
print(df.head())


['log_id', 'customer_id', 'event_time', 'event_type', 'product_id', 'session_id']
   log_id  customer_id           event_time event_type  product_id  \
0       1       1946.0  2024-03-05 05:36:18       view        36.0   
1       2       2132.0  2024-06-29 11:33:41   purchase       448.0   
2       3          NaN  2024-02-29 01:12:19       view       301.0   
3       4          NaN  2024-01-24 14:03:25       view       308.0   
4       5       3801.0  2024-04-06 01:10:29       view       177.0   

      session_id  
0  sess_001e5511  
1  sess_b60fb67c  
2  sess_d09e432b  
3  sess_5cc9f9b6  
4  sess_029b23c8  


In [ ]:
# 세션 x 이벤트유형 "도달 여부" 행렬
# 한행은 한번 발생한 이벤트 -> 이 뜻은 같은 세션에서 여러행동(view하고 search했으면 session_id가 반복됨)

# - 한 세션에 같은 이벤트가 여러 번 찍히므로, 
# 같은 세션의 같은 이벤트 중복 제거
sess_event = logs.drop_duplicates(subset=["session_id", "event_type"])

# 세션을 row, 이벤트를 column으로 바꾸기
# crosstab()으로 교차표를 만들기
reach = pd.crosstab(
    sess_event["session_id"],   #row
    sess_event["event_type"]    #column
)

#값을 확실하게 0과 1로 바꾸기 -> True/False int(1 과 0으로 바꾸기)
reach = (reach > 0).astype(int)

#reach.shape 은 dataframe의 모양을 (행 개수, 열 개수) 로 반환함
#reach.shape[0]은 row 개수
print("전체 세션 수:", reach.shape[0])
reach.sum() #각 이벤트에 도달한 세션수 

전체 세션 수: 245430


event_type
cart         94867
purchase     68770
search      137867
view        227346
dtype: int64

### 퍼널 단계 정의 규칙 (핵심 판단)

1. **도달 여부 기준**: view/cart/purchase 각각 세션 내 발생 횟수가 아니라 "해당 이벤트가 한 번이라도 있었는가"(0/1)로만 판단한다.
2. **정규 경로 = view ∩ cart ∩ purchase**: "view→cart→purchase 전체 전환율"은 세 이벤트를 모두 겪은 세션만 분자로 인정한다. (이벤트 시각 순서까지는 강제하지 않고, 세션 내 3개 이벤트 동시 도달 여부로 완화하여 정의 — 실무 퍼널 집계에서 흔히 쓰는 방식)
3. **cart 없이 purchase만 있는 세션 = "다이렉트 구매"로 분리 집계**: 정규 경로 전환율의 분자에서는 제외하고, 별도 지표로만 보고한다. (원인 후보: 즉시구매/원클릭 구매 버튼, 재구매 딥링크 등 — 이번 과제 범위 밖이므로 규모만 확인)
4. **view 없이 cart/purchase만 있는 세션**은 로그 유실·세션 매칭 이슈로 보고 정규 경로 분석 대상에서 제외한다(건수만 각주로 표기).


In [ ]:
# 규칙에 따른 세션 수 산출

#퍼널 단계별 세션수와 전환율을 계산하는 코드
n_view = int(reach["view"].sum())  #상품 조회 이벤트가 한번이라도 있었던 세션수
#view를 한 세션 & cart 도 한세션  (&는 두조건이 모두 참)
n_view_cart = int(((reach["view"] ==1 ) & (reach["cart"] == 1)).sum())
#view, cart, purchase한 세션수
n_full_path = int(((reach["view"] == 1) & (reach["cart"] == 1) & (reach["purchase"] == 1)).sum())

#전체 구매 세션 수  - 구매가 한번이라도 발생한 전체 세션수
n_purchase_total = int(reach["purchase"].sum())

# 참고용 이상 경로 계산: 장바구니 없이 바로 구매, view 없이 cart만, view 없이 구매 등
n_direct_purchase = int(((reach["purchase"] == 1) & (reach["cart"] == 0)).sum())
n_cart_no_view = int(((reach["cart"] == 1) & (reach["view"] == 0)).sum())
n_purchase_no_view = int(((reach["purchase"] == 1) & (reach["view"] == 0)).sum())

#퍼널 결과표
funnel = pd.DataFrame({
    "stage": ["view", "cart", "purchase"],
    "sessions": [n_view, n_view_cart, n_full_path],
})
funnel["step_conversion"] = funnel["sessions"] / funnel["sessions"].shift(1)
funnel["conversion_from_view"] = funnel["sessions"] / n_view
funnel

,stage,sessions,step_conversion,conversion_from_view
0,view,227346,NaN,1.000000
1,cart,86193,0.379127,0.379127
2,purchase,23687,0.274813,0.104189


In [6]:
# 참고 지표 출력 (정규 경로 전환율 분자에는 포함하지 않음)
print(f"다이렉트 구매(cart 미경유) 세션: {n_direct_purchase:,} / 전체 구매 세션 {n_purchase_total:,} ({n_direct_purchase/n_purchase_total:.1%})")
print(f"view 없이 cart만 있는 세션: {n_cart_no_view:,}")
print(f"view 없이 purchase만 있는 세션: {n_purchase_no_view:,}")

다이렉트 구매(cart 미경유) 세션: 42,818 / 전체 구매 세션 68,770 (62.3%)
view 없이 cart만 있는 세션: 8,674
view 없이 purchase만 있는 세션: 6,177


### 병목 진단

| 단계 | 세션 수 | 단계 전환율 | view 기준 누적 전환율 |
|---|---:|---:|---:|
| view | 227,346 | - | 100% |
| cart | 86,193 | 37.9% | 37.9% |
| purchase(정규 경로) | 23,687 | 27.5% | 10.4% |

- **가장 큰 이탈 단계는 cart → purchase다.** 이 단계의 전환율(27.5%)이 view → cart(37.9%)보다 낮아, 장바구니에 담고도 결제로 못 넘어가는 세션이 62,506건(86,193 − 23,687)으로 절대량도 가장 크다.
- 다만 view → cart 단계도 이탈량 자체는 141,153건으로 규모가 크므로, 실험은 두 단계 모두를 후보로 두되 **전환율이 더 낮은 cart → purchase를 1순위**로 잡는다.
- 참고: 전체 구매(68,770건) 중 다이렉트 구매(cart 미경유)가 42,818건(62.3%)이나 된다. 이는 정규 경로 밖의 현상이라 이번 병목 진단에서는 제외했지만, "원클릭/즉시구매" 경로가 이미 상당한 매출을 만들고 있다는 뜻이므로 별도 트래킹이 필요하다.

### 개선 실험 아이디어

1. **체크아웃 마찰 축소 A/B 테스트** — 장바구니→결제 단계 수를 줄이고(배송지/결제수단 입력 통합), 게스트 체크아웃과 간편결제(카카오페이 등)를 노출해 cart → purchase 전환율 변화를 측정한다.
2. **장바구니 이탈 리마인드 실험** — cart 도달 후 미구매 세션을 대상으로 무료배송 임계값 안내 배너·재고 임박 카운트다운을 장바구니 페이지에 노출하는 실험군과, 미노출 대조군을 나눠 cart → purchase 회복률을 비교한다.
